# 02 — Historico de fichajes de Rayados

Bloque #1 del trabajo sucio.

**Objetivo:** descargar entradas y salidas de los ultimos 8 anios (2017/18 a 2024/25) desde Transfermarkt.

**Tiempo estimado:** 1-2 min (8 paginas, 5 seg de pausa entre cada una).

**Estrategia:** primero scrapear 1 sola temporada (2023/24) para validar el parser. Si funciona, escalar a las 8.

In [ ]:
# Celda 1 — Setup
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import data_fetcher as df_mod
import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

print(f'Project root: {PROJECT_ROOT}')
print(f'Cache dir:    {df_mod.DATA_RAW}')

## Paso 1: Probar con UNA temporada

Antes de descargar 8, validamos el parser con la temporada 2023/24 que es reciente y rica.

In [ ]:
# Celda 2 — Probar 2023/24
from src.data_fetcher import fetch_tm_transfers_single_season

try:
    df_2023 = fetch_tm_transfers_single_season(
        club_id=df_mod.CLUB_RAYADOS_TM_ID,
        season_id=2023,
        force_refresh=True,
    )
    print(f'\n[OK] {len(df_2023)} fichajes extraidos')
    print(f'\nDirecciones detectadas: {df_2023["direccion"].value_counts().to_dict()}')
    print(f'\nPrimeras filas:')
    print(df_2023.head(10).to_string(index=False))
except RuntimeError as e:
    print(f'[ERROR] {e}')
    print('\nGuardando HTML para debug...')
    import requests
    url = f'{df_mod.TM_BASE}/cf-monterrey/transfers/verein/{df_mod.CLUB_RAYADOS_TM_ID}/saison_id/2023'
    r = requests.get(url, headers=df_mod.HEADERS_BROWSER, timeout=30)
    df_mod.save_debug_html(r.text, 'debug_tm_transfers_2023.html')
    print('Adjunta data/raw/debug_tm_transfers_2023.html a Claude')

## Paso 2: Si el paso 1 funciona, escalar a 8 temporadas

NO ejecutes esta celda hasta verificar que el paso 1 dio datos coherentes.

In [ ]:
# Celda 3 — Descargar 8 temporadas
from src.data_fetcher import fetch_tm_transfers_multi_seasons

df_all = fetch_tm_transfers_multi_seasons(
    club_id=df_mod.CLUB_RAYADOS_TM_ID,
    seasons=list(range(2017, 2025)),   # 2017/18 a 2024/25
    force_refresh=False,                # usar cache de lo ya descargado
)

print(f'\n[OK] {len(df_all)} fichajes totales en 8 temporadas')
print(f'\nDirecciones: {df_all["direccion"].value_counts().to_dict()}')
print(f'Por temporada:')
print(df_all.groupby(["season_id", "direccion"]).size().unstack(fill_value=0))

## Paso 3: Inspeccion de calidad

Verificar que los datos tienen sentido antes de pasar al processing.

In [ ]:
# Celda 4 — Sanity checks
if 'df_all' in dir() and df_all is not None and not df_all.empty:
    print('Top 10 fichajes mas caros (LLEGADAS):')
    top_in = df_all[df_all['direccion'] == 'in'].nlargest(10, 'coste_meur')
    print(top_in[['season_id','jugador','edad','posicion_tm','club_relacionado','coste_meur']].to_string(index=False))
    
    print('\nTop 10 ventas mas caras (SALIDAS):')
    top_out = df_all[df_all['direccion'] == 'out'].nlargest(10, 'coste_meur')
    print(top_out[['season_id','jugador','edad','posicion_tm','club_relacionado','coste_meur']].to_string(index=False))
    
    print('\nLigas de origen mas frecuentes (entradas):')
    print(df_all[df_all['direccion']=='in']['liga_relacionada'].value_counts().head(10))